<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_1/lessons/lesson_06_dicts_loops_comprehensions/note_lesson_06_dicts_loops_comprehensions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Урок 6 — Словники, for, comprehensions

> За цей урок: після списків/кортежів/множин (Урок 5) додаємо останній базовий контейнер — `dict`, п'ять патернів мислення для обробки колекцій (Iteration/Mapping/Aggregation/Counting/Grouping), і компактнішу форму запису циклу — comprehensions. Усе це одразу застосовуємо до реальних даних — 244 чеків ресторану: виторг за днями, найприбутковіший день, групування і фільтри. Урок 8 розкладе такий звіт на функції.

Структура уроку: **RETRIEVE → CONCEPT → аналітика ресторану → TRANSFER** (та сама послідовність, що й в Уроках 3, 4, 5, 8).

## 🔁 RETRIEVE — пригадай Урок 5 (без підглядання)

1. Чим `tuple` принципово відрізняється від `list` — не за синтаксисом, а за призначенням?
2. Що поверне `{1, 2, 2, 3}` — скільки елементів у результаті?
3. Які три частини потрібні, щоб пройти список чеків циклом `while` і порахувати суму?

<details>
<summary>Відповіді</summary>

1. <code>tuple</code> незмінний — використовується для запису, що вже зафіксований (факт, що стався) і не повинен випадково змінитись; <code>list</code> — для послідовності, яка змінюється по ходу програми.
2. <code>3</code> елементи — <code>set</code> автоматично видаляє дублікати, лишається <code>{1, 2, 3}</code>.
3. Лічильник-індекс <code>i = 0</code>, умова <code>i &lt; len(orders)</code> і крок <code>i += 1</code> наприкінці тіла (плюс накопичувач <code>total = 0</code>, до якого додаємо суму кожного чека). Цикл <code>for</code> з цього уроку бере індекс на себе.

</details>

## 📖 CONCEPT

### 1. `dict` — четвертий контейнер: доступ за ключем, а не за позицією

У списку елемент шукають за **позицією** (`fruits[0]`). У словнику — за **ключем**, будь-яким незмінним значенням (найчастіше рядком):

In [1]:
student = {"name": "Олена", "age": 20, "group": "КН-21"}

print(student["name"])   # доступ за ключем
print(student)

# Звернення до відсутнього ключа — помилка, так само як IndexError у списку за межами
try:
    print(student["email"])
except KeyError as e:
    print(f"KeyError: {e}")

Олена
{'name': 'Олена', 'age': 20, 'group': 'КН-21'}
KeyError: 'email'


### 2. `.get()` — безпечний доступ

`d[key]` кидає `KeyError`, якщо ключа немає. `d.get(key, default)` замість помилки повертає запасне значення:

In [2]:
print(student.get("email"))            # ключа немає -> None (default за замовчуванням)
print(student.get("email", "немає"))   # ключа немає -> "немає"
print(student.get("name", "немає"))    # ключ є -> звичайне значення, default ігнорується

None
немає
Олена


### 3. Ітерація по словнику — три способи, різний результат

```
for key in d:          # тільки ключі
for val in d.values():  # тільки значення
for k, v in d.items():  # пари (ключ, значення)
```

**Типова пастка:** якщо написати `for k, v in d` (без `.items()`), Python спробує розпакувати сам **ключ** (рядок) у дві змінні `k, v` — і це впаде з `ValueError`, якщо ключ не рівно 2 символи. Подивимось на це напряму:

In [3]:
scores = {"Alice": 95, "Bob": 82, "Cara": 77}

print("for key in d:")
for key in scores:
    print(" ", key)

print()
print("for val in d.values():")
for val in scores.values():
    print(" ", val)

print()
print("for k, v in d.items():")
for k, v in scores.items():
    print(" ", k, "->", v)

print()
print("Пастка — for k, v in d (без .items()):")
try:
    for k, v in scores:
        print(k, v)
except ValueError as e:
    print(f"  ValueError: {e}")
    print('  Причина: for key in d дає рядки типу "Alice" (5 символів) — розпакувати їх у 2 змінні не можна.')

for key in d:
  Alice
  Bob
  Cara

for val in d.values():
  95
  82
  77

for k, v in d.items():
  Alice -> 95
  Bob -> 82
  Cara -> 77

Пастка — for k, v in d (без .items()):
  ValueError: too many values to unpack (expected 2)
  Причина: for key in d дає рядки типу "Alice" (5 символів) — розпакувати їх у 2 змінні не можна.


### 4. `.setdefault()` — читання з автоматичним створенням

`d.setdefault(key, default)`: якщо ключ **є** — просто повертає його значення (нічого не змінює). Якщо ключа **нема** — створює `d[key] = default` і одразу повертає це `default`. Найкорисніше, коли значення — список, який ми одразу наповнюємо:

In [4]:
teams = {}

teams.setdefault("Backend", []).append("Олег")
teams.setdefault("Backend", []).append("Ірина")   # ключ вже є -> список не перестворюється
teams.setdefault("Frontend", []).append("Марко")

print(teams)
# {'Backend': ['Олег', 'Ірина'], 'Frontend': ['Марко']}

{'Backend': ['Олег', 'Ірина'], 'Frontend': ['Марко']}


### 5. Патерн Counting — підрахунок частоти

Ідея: перебираємо елементи, для кожного збільшуємо лічильник у словнику `{значення: скільки_разів}`. `dict.get(key, 0) + 1` — стандартний прийом "прочитати поточне значення, або 0, якщо ще не було":

In [5]:
text = "banana"
counts = {}

for char in text:
    counts[char] = counts.get(char, 0) + 1   # був 0 -> стає 1, був N -> стає N+1

print(counts)
# {'b': 1, 'a': 3, 'n': 2}

# Найчастіший символ — max() з ключем "за яким значенням шукати максимум"
most_common = max(counts, key=counts.get)
print(f"Найчастіший символ: {most_common!r} ({counts[most_common]} разів)")

{'b': 1, 'a': 3, 'n': 2}
Найчастіший символ: 'a' (3 разів)


### 6. Патерн Grouping — групування записів за ключем

Розширення Counting: замість того щоб просто рахувати, збираємо самі значення у списки за ключем — `.setdefault(key, []).append(item)`:

In [6]:
numbers = [4, 15, 22, 7, 30, 3, 18, 9]

by_parity = {}
for n in numbers:
    key = "парні" if n % 2 == 0 else "непарні"
    by_parity.setdefault(key, []).append(n)

print(by_parity)
# {'парні': [4, 22, 30, 18], 'непарні': [15, 7, 3, 9]}

{'парні': [4, 22, 30, 18], 'непарні': [15, 7, 3, 9]}


### 7. `for` проти `while` — та сама задача, дві мови

`for` керується **колекцією** (Python сам бере наступний елемент). `while` керується **умовою** (ви самі відповідаєте, коли зупинитись). Та сама задача (пройти по трьох елементах) обома способами:

In [7]:
letters = ["c", "a", "t"]

print("for (Python сам керує):")
for letter in letters:
    print(" ", letter)

print()
print("while (керуємо самі, легко забути i += 1 -> нескінченний цикл):")
i = 0
while i < len(letters):
    print(" ", letters[i])
    i += 1

for (Python сам керує):
  c
  a
  t

while (керуємо самі, легко забути i += 1 -> нескінченний цикл):
  c
  a
  t


### 8. List comprehension — той самий цикл, компактніший запис

Спочатку задача звичайним циклом: **квадрати парних чисел від 1 до 10**:

In [8]:
even_squares = []
for x in range(1, 11):
    if x % 2 == 0:
        even_squares.append(x ** 2)

print(even_squares)
# [4, 16, 36, 64, 100]

[4, 16, 36, 64, 100]


Той самий результат, одним рядком — **та сама ідея**, лише компактніше записана:

```
[ що_зберегти   for елемент in колекція   if умова ]
```

In [9]:
even_squares_v2 = [x ** 2 for x in range(1, 11) if x % 2 == 0]

print(even_squares_v2)
assert even_squares_v2 == even_squares
print("Той самий результат, що й циклом вище")

[4, 16, 36, 64, 100]
Той самий результат, що й циклом вище


Comprehension може бути й без фільтра (тільки трансформація), і без трансформації (тільки фільтр):

In [10]:
squares = [x ** 2 for x in range(1, 6)]                # тільки трансформація
positives = [x for x in [-3, 5, -1, 8, 0, -7] if x > 0]  # тільки фільтр

print(squares)
print(positives)

# Правило: comprehension будує НОВИЙ список. Якщо мета — виконати дію (print,
# зберегти у файл) без побудови списку — використовуйте звичайний for.

[1, 4, 9, 16, 25]
[5, 8]


### 9. Dict comprehension — та сама ідея для словників

Знову спочатку звичайним циклом: словник `{слово: довжина_слова}`:

In [11]:
words = ["apple", "banana", "cherry", "date"]

word_lengths = {}
for w in words:
    word_lengths[w] = len(w)

print(word_lengths)
# {'apple': 5, 'banana': 6, 'cherry': 6, 'date': 4}

{'apple': 5, 'banana': 6, 'cherry': 6, 'date': 4}


In [12]:
word_lengths_v2 = {w: len(w) for w in words}

print(word_lengths_v2)
assert word_lengths_v2 == word_lengths
print("Той самий результат, що й циклом вище")

{'apple': 5, 'banana': 6, 'cherry': 6, 'date': 4}
Той самий результат, що й циклом вище


### Підсумок CONCEPT

| Патерн | Питання | Інструмент |
|---|---|---|
| Iteration | Як пройтись по всіх елементах? | `for` / `while` |
| Mapping | Як перетворити кожен елемент? | comprehension |
| Counting | Скільки разів зустрівся кожен варіант? | `d.get(key, 0) + 1` |
| Grouping | Які саме записи в кожній групі? | `d.setdefault(key, []).append(...)` |

Далі — застосовуємо саме ці інструменти (`dict`, `set`, `for`, comprehensions) до реальних чеків ресторану.

### Шпаргалка

```python
# dict
d[key]                    # KeyError, якщо немає
d.get(key, default)       # безпечне читання
d.setdefault(key, default)  # читання + створення, якщо немає
for k in d: ...            # тільки ключі
for k, v in d.items(): ... # пари

# Counting / Grouping
d[key] = d.get(key, 0) + 1          # підрахунок
d.setdefault(key, []).append(item)  # групування

# Comprehensions
[expr for x in seq if cond]   # list
{k: v for k, v in seq}        # dict
{expr for x in seq}           # set
```

## 🍽️ Наскрізний приклад: аналітика ресторану

У попередньому уроці (позиція 5 — списки, кортежі, множини) ми перетворили 244 реальних чеки ресторану на `orders: List[Order]`, де `Order` — це той самий `NamedTuple`, що й в уроці 5 (`waiter, total_bill, tip, day, time, size`), плюс дві колонки датасету `sex` і `smoker`, які знадобляться в задачах. Тут ми продовжуємо саме з цієї точки — агрегуємо ці чеки словниками, шукаємо лідера, групуємо і фільтруємо через comprehensions. Модель мислення та сама, що й у CONCEPT вище: дані → накопичення в dict → результат.

Коротко повторюємо з попереднього уроку, щоб продовжити тут:

In [13]:
import seaborn as sns
from typing import NamedTuple


class Order(NamedTuple):
    waiter: str          # той самий Order, що в уроці 5: перше поле — хто обслуговував
    total_bill: float
    tip: float
    day: str
    time: str
    size: int
    sex: str             # дві додаткові колонки датасету — потрібні в задачі 2
    smoker: str


tips_df = sns.load_dataset("tips")

# Офіціанта в датасеті немає — як і в уроці 5, приймаємо: одна зміна (день + час) = один офіціант
orders = []
for _, row in tips_df.iterrows():
    day, time = str(row["day"]), str(row["time"])
    if (day, time) in (("Thur", "Lunch"), ("Fri", "Lunch")):
        waiter = "Марія"
    elif (day, time) in (("Thur", "Dinner"), ("Sat", "Dinner")):
        waiter = "Тарас"
    else:                                    # Fri Dinner, Sun Dinner
        waiter = "Олексій"
    orders.append(Order(waiter, float(row["total_bill"]), float(row["tip"]), day, time, int(row["size"]),
                        str(row["sex"]), str(row["smoker"])))

print(f"Завантажено: {len(orders)} чеків")
print(orders[0])

Завантажено: 244 чеків
Order(waiter='Олексій', total_bill=16.99, tip=1.01, day='Sun', time='Dinner', size=2, sex='Female', smoker='No')


### `for` як обробник потоку подій

Кожна ітерація `for order in orders` — це один клієнт, що заплатив рахунок. Спочатку простий прохід (дивимось на дані), потім накопичувач (Aggregation pattern), як `order_total += price` з уроку 4, тільки тепер без ручного індексу:

In [14]:
print("Перші 5 чеків:")
for order in orders[:5]:
    tip_pct = (order.tip / order.total_bill) * 100
    print(f"{order.day:6} | {order.time:7} | ${order.total_bill:6.2f} | tip {tip_pct:.1f}%")

print()

total_revenue = 0
total_tips = 0
total_guests = 0

for order in orders:
    total_revenue += order.total_bill
    total_tips += order.tip
    total_guests += order.size

avg_bill = total_revenue / len(orders)

print(f"Загальний дохід:  ${total_revenue:.2f}")
print(f"Загальні чайові:  ${total_tips:.2f}")
print(f"Всього гостей:    {total_guests}")
print(f"Середній чек:     ${avg_bill:.2f}")

Перші 5 чеків:
Sun    | Dinner  | $ 16.99 | tip 5.9%
Sun    | Dinner  | $ 10.34 | tip 16.1%
Sun    | Dinner  | $ 21.01 | tip 16.7%
Sun    | Dinner  | $ 23.68 | tip 14.0%
Sun    | Dinner  | $ 24.59 | tip 14.7%

Загальний дохід:  $4827.77
Загальні чайові:  $731.58
Всього гостей:    627
Середній чек:     $19.79


### Counting pattern: `d[key] = d.get(key, 0) + value`

Найпоширеніший алгоритм бізнес-аналітики. Спочатку побачимо, чому наївний варіант падає:

In [15]:
# Наївний варіант — падає, бо ключа "apple" ще немає
fruits = {}
try:
    fruits["apple"] = fruits["apple"] + 1
except KeyError as e:
    print(f"KeyError: {e} — ключа ще немає, нема що додавати")

# Виправлено: .get(key, 0) підставляє 0, якщо ключа немає
fruits = {}
fruits["apple"] = fruits.get("apple", 0) + 1
print(fruits)

KeyError: 'apple' — ключа ще немає, нема що додавати
{'apple': 1}


Живий перегляд — як словник оновлюється на перших 10 чеках, крок за кроком:

In [16]:
revenue_by_day = {}

print("Крок | День  | Чек     | Стан dict")
for i, order in enumerate(orders[:10], 1):
    day = order.day
    bill = order.total_bill
    revenue_by_day[day] = revenue_by_day.get(day, 0) + bill
    state = {k: f"${v:.0f}" for k, v in revenue_by_day.items()}
    print(f"  {i:2} | {day:5} | ${bill:6.2f} | {state}")

Крок | День  | Чек     | Стан dict
   1 | Sun   | $ 16.99 | {'Sun': '$17'}
   2 | Sun   | $ 10.34 | {'Sun': '$27'}
   3 | Sun   | $ 21.01 | {'Sun': '$48'}
   4 | Sun   | $ 23.68 | {'Sun': '$72'}
   5 | Sun   | $ 24.59 | {'Sun': '$97'}
   6 | Sun   | $ 25.29 | {'Sun': '$122'}
   7 | Sun   | $  8.77 | {'Sun': '$131'}
   8 | Sun   | $ 26.88 | {'Sun': '$158'}
   9 | Sun   | $ 15.04 | {'Sun': '$173'}
  10 | Sun   | $ 14.78 | {'Sun': '$187'}


Тепер по всіх 244 чеках — три словники за один цикл (revenue, кількість чеків, гостей по днях):

In [17]:
revenue_by_day = {}
orders_by_day = {}
guests_by_day = {}

for order in orders:
    day = order.day
    revenue_by_day[day] = revenue_by_day.get(day, 0) + order.total_bill
    orders_by_day[day] = orders_by_day.get(day, 0) + 1
    guests_by_day[day] = guests_by_day.get(day, 0) + order.size

print("День  | Дохід     | Чеків | Гостей")
for day in revenue_by_day:
    print(f"{day:6} | ${revenue_by_day[day]:8.2f} | {orders_by_day[day]:5} | {guests_by_day[day]:6}")

assert sum(orders_by_day.values()) == len(orders)
assert abs(sum(revenue_by_day.values()) - total_revenue) < 0.01
print("\nOK — сума по днях збігається із загальним доходом і кількістю чеків")

День  | Дохід     | Чеків | Гостей
Sun    | $ 1627.16 |    76 |    216
Sat    | $ 1778.40 |    87 |    219
Thur   | $ 1096.33 |    62 |    152
Fri    | $  325.88 |    19 |     40

OK — сума по днях збігається із загальним доходом і кількістю чеків


### Leader algorithm: який день найприбутковіший?

Ручний алгоритм «поточний чемпіон», потім пітонічний варіант через `max(d, key=d.get)` — і перевіряємо, що обидва дають той самий результат:

In [18]:
# Варіант 1 — ручний алгоритм «поточний чемпіон»
best_day_manual = None
best_revenue_manual = 0
for day, revenue in revenue_by_day.items():
    if revenue > best_revenue_manual:
        best_revenue_manual = revenue
        best_day_manual = day

# Варіант 2 — пітонічний через max()
best_day = max(revenue_by_day, key=revenue_by_day.get)

assert best_day == best_day_manual
print(f"Найкращий день: {best_day} — ${revenue_by_day[best_day]:.2f}")

days_sorted = sorted(revenue_by_day.items(), key=lambda x: x[1], reverse=True)
print("\nРейтинг днів:")
for i, (day, rev) in enumerate(days_sorted, 1):
    bar = "#" * int(rev / 80)
    print(f"  {i}. {day:6}: ${rev:7.2f}  {bar}")

Найкращий день: Sat — $1778.40

Рейтинг днів:
  1. Sat   : $1778.40  ######################
  2. Sun   : $1627.16  ####################
  3. Thur  : $1096.33  #############
  4. Fri   : $ 325.88  ####


### Той самий Counting/Leader патерн — інший ключ

Той самий алгоритм, застосований до `order.time` замість `order.day`, без жодної зміни логіки — тільки поле, по якому групуємо:

In [19]:
revenue_by_time = {}
orders_by_time = {}

for order in orders:
    t = order.time
    revenue_by_time[t] = revenue_by_time.get(t, 0) + order.total_bill
    orders_by_time[t] = orders_by_time.get(t, 0) + 1

print("Час     | Дохід      | Чеків | Avg чек")
for t in revenue_by_time:
    n = orders_by_time[t]
    rev = revenue_by_time[t]
    print(f"{t:7} | ${rev:9.2f} | {n:5} | ${rev/n:7.2f}")

winner = max(revenue_by_time, key=revenue_by_time.get)
assert abs(sum(revenue_by_time.values()) - total_revenue) < 0.01
print(f"\nПереможець: {winner}")

Час     | Дохід      | Чеків | Avg чек
Dinner  | $  3660.30 |   176 | $  20.80
Lunch   | $  1167.47 |    68 | $  17.17

Переможець: Dinner


### `set` — унікальні значення без ручної перевірки

Ручний збір через `set.add()`, потім та сама ідея через set comprehension (з CONCEPT вище) — і перевіряємо рівність:

In [20]:
unique_days_manual = set()
for order in orders:
    unique_days_manual.add(order.day)

unique_days = {order.day for order in orders}
unique_times = {order.time for order in orders}
unique_sizes = {order.size for order in orders}

assert unique_days == unique_days_manual
print("Дні роботи:    ", unique_days)
print("Типи зміни:    ", unique_times)
print("Розміри столів:", sorted(unique_sizes))

Дні роботи:     {'Sun', 'Fri', 'Thur', 'Sat'}
Типи зміни:     {'Lunch', 'Dinner'}
Розміри столів: [1, 2, 3, 4, 5, 6]


### Grouping pattern: `dict.setdefault(key, []).append(item)`

Різниця з Counting: не «скільки», а «які саме» — зберігаємо самі об'єкти `Order`, не лише лічильник:

In [21]:
orders_by_day = {}
for order in orders:
    orders_by_day.setdefault(order.day, []).append(order)

print("День  | Чеків | Avg чек | Max чек")
for day, day_orders in orders_by_day.items():
    bills = [o.total_bill for o in day_orders]
    print(f"{day:6} | {len(day_orders):5} | ${sum(bills)/len(bills):6.2f}  | ${max(bills):.2f}")

assert sum(len(v) for v in orders_by_day.values()) == len(orders)
assert set(orders_by_day.keys()) == unique_days
print("\nOK — сума груп дорівнює кількості чеків, ключі збігаються з унікальними днями")

День  | Чеків | Avg чек | Max чек
Sun    |    76 | $ 21.41  | $48.17
Sat    |    87 | $ 20.44  | $50.81
Thur   |    62 | $ 17.68  | $43.11
Fri    |    19 | $ 17.15  | $40.17

OK — сума груп дорівнює кількості чеків, ключі збігаються з унікальними днями


### `defaultdict` — той самий алгоритм, чистіший синтаксис

`defaultdict(int)`/`defaultdict(float)`/`defaultdict(list)` автоматично підставляють значення за замовчуванням для нового ключа — не треба ні `if`, ні `.get()`, ні `.setdefault()`. Переписуємо всі три патерни й перевіряємо, що результат ідентичний тому, що вже порахували вище:

In [22]:
from collections import defaultdict

# Counting — раніше: orders_by_time[t] = orders_by_time.get(t, 0) + 1
orders_by_time_dd = defaultdict(int)
for order in orders:
    orders_by_time_dd[order.time] += 1

# Aggregation — раніше: revenue_by_day[day] = revenue_by_day.get(day, 0) + bill
revenue_by_day_dd = defaultdict(float)
for order in orders:
    revenue_by_day_dd[order.day] += order.total_bill

# Grouping — раніше: orders_by_day.setdefault(day, []).append(order)
groups = defaultdict(list)
for order in orders:
    groups[order.day].append(order)

assert dict(orders_by_time_dd) == orders_by_time
assert {k: round(v, 2) for k, v in revenue_by_day_dd.items()} == {k: round(v, 2) for k, v in revenue_by_day.items()}
assert {k: len(v) for k, v in groups.items()} == {k: len(v) for k, v in orders_by_day.items()}
print("OK — defaultdict дає той самий результат, що й .get()/.setdefault() варіанти")

OK — defaultdict дає той самий результат, що й .get()/.setdefault() варіанти


### `Counter` — третій спосіб рахувати

Коли єдине, що потрібно — це «скільки разів щось зустрілось», `collections.Counter` ще компактніший за `defaultdict(int)`: приймає готову послідовність і одразу рахує, плюс дає `.most_common()`:

In [23]:
from collections import Counter

orders_count_counter = Counter(order.day for order in orders)

print("Кількість чеків по днях (найпопулярніші перші):")
for day, count in orders_count_counter.most_common():
    print(f"  {day:6}: {count} чеки")

# Порівнюємо з тим самим підрахунком через .get() — мають збігтись
orders_count_get = {}
for order in orders:
    orders_count_get[order.day] = orders_count_get.get(order.day, 0) + 1

assert dict(orders_count_counter) == orders_count_get
print("\nOK — Counter і .get() дають однаковий підрахунок по днях")

Кількість чеків по днях (найпопулярніші перші):
  Sat   : 87 чеки
  Sun   : 76 чеки
  Thur  : 62 чеки
  Fri   : 19 чеки

OK — Counter і .get() дають однаковий підрахунок по днях


### Comprehensions на реальних даних: фільтр, трансформація, топ-N

Застосовуємо list comprehensions (з CONCEPT вище) до `orders` — фільтр, мапінг, і фільтр+трансформація разом:

In [24]:
# Filter: тільки умова
big_orders = [o for o in orders if o.total_bill > 40]
print(f"Чеків понад $40: {len(big_orders)}")

# Mapping: трансформація (tip% для кожного чеку)
all_tip_percents = [(o.tip / o.total_bill) * 100 for o in orders]
avg_tip = sum(all_tip_percents) / len(all_tip_percents)
print(f"Tip %: avg={avg_tip:.1f}%  min={min(all_tip_percents):.1f}%  max={max(all_tip_percents):.1f}%")

# Filter + Transform
dinner_bills = [o.total_bill for o in orders if o.time == "Dinner"]
print(f"Dinner чеків: {len(dinner_bills)}, avg ${sum(dinner_bills)/len(dinner_bills):.2f}")

# Топ-10 найбільших чеків — sorted() + slice
top10 = sorted(orders, key=lambda o: o.total_bill, reverse=True)[:10]
print("\nТоп-3 з топ-10:")
for o in top10[:3]:
    print(f"  {o.day:6} {o.time:7} ${o.total_bill:6.2f}")

assert len(big_orders) + len([o for o in orders if o.total_bill <= 40]) == len(orders)
assert len(dinner_bills) == orders_by_time["Dinner"]
print("\nOK — фільтр + доповнення до нього дають повну множину чеків")

Чеків понад $40: 10
Tip %: avg=16.1%  min=3.6%  max=71.0%
Dinner чеків: 176, avg $20.80

Топ-3 з топ-10:
  Sat    Dinner  $ 50.81
  Sat    Dinner  $ 48.33
  Sat    Dinner  $ 48.27

OK — фільтр + доповнення до нього дають повну множину чеків


### Dict comprehension: середній чек і середні чайові по днях

Спочатку звичайний цикл (для перевірки), потім dict comprehension — і порівнюємо, що результат той самий:

In [25]:
# Кількість чеків по днях (Counting — уже знайомий патерн)
count_by_day = {}
for order in orders:
    count_by_day[order.day] = count_by_day.get(order.day, 0) + 1

# Звичайним циклом
avg_bill_by_day_loop = {}
for day in revenue_by_day:
    avg_bill_by_day_loop[day] = revenue_by_day[day] / count_by_day[day]

# Dict comprehension — та сама ідея, один вираз
avg_bill_by_day = {day: revenue_by_day[day] / count_by_day[day] for day in revenue_by_day}

assert avg_bill_by_day == avg_bill_by_day_loop
print("Середній чек по днях:")
for day, avg in avg_bill_by_day.items():
    print(f"  {day:6}: ${avg:.2f}")

Середній чек по днях:
  Sun   : $21.41
  Sat   : $20.44
  Thur  : $17.68
  Fri   : $17.15


### Dict comprehension: середні чайові (%) по днях

Використовуємо `groups` (`defaultdict(list)` з попереднього кроку) — для кожного дня беремо список чеків і рахуємо середній tip% спочатку циклом, потім comprehension:

In [26]:
tip_pct_by_day_loop = {}
for day, day_orders in groups.items():
    tip_pct_by_day_loop[day] = sum((o.tip / o.total_bill) * 100 for o in day_orders) / len(day_orders)

tip_pct_by_day = {
    day: sum((o.tip / o.total_bill) * 100 for o in day_orders) / len(day_orders)
    for day, day_orders in groups.items()
}

assert tip_pct_by_day.keys() == tip_pct_by_day_loop.keys()
for day in tip_pct_by_day:
    assert abs(tip_pct_by_day[day] - tip_pct_by_day_loop[day]) < 1e-9

print("Середній % чайових по днях:")
for day, pct in sorted(tip_pct_by_day.items(), key=lambda x: x[1], reverse=True):
    print(f"  {day:6}: {pct:.2f}%")

Середній % чайових по днях:
  Fri   : 16.99%
  Sun   : 16.69%
  Thur  : 16.13%
  Sat   : 15.32%


### Фінальний звіт: зводимо все докупи

Один прохід по вже готових агрегатах (`revenue_by_day`, `total_revenue`, `total_tips`, `total_guests`, `avg_bill`) плюс два нові фільтри (обід/вечеря, розмір столу) — усе тим самим патерном, що й вище, зібране у формат «звіт для власника»:

In [27]:
print("=" * 60)
print("        ЗВІТ ДЛЯ ВЛАСНИКА РЕСТОРАНУ")
print("=" * 60)

avg_tip_pct = sum((o.tip / o.total_bill) * 100 for o in orders) / len(orders)

print(f"\nЗАГАЛЬНА СТАТИСТИКА")
print(f"  Всього чеків:         {len(orders)}")
print(f"  Всього гостей:        {total_guests}")
print(f"  Загальний дохід:      ${total_revenue:.2f}")
print(f"  Загальні чайові:      ${total_tips:.2f}")
print(f"  Середній чек:         ${avg_bill:.2f}")
print(f"  Середній tip %:       {avg_tip_pct:.1f}%")

days_rank = sorted(revenue_by_day.items(), key=lambda x: x[1], reverse=True)
print(f"\nНАЙКРАЩІ ДНІ")
for i, (day, rev) in enumerate(days_rank, 1):
    marker = "  <- найкращий!" if i == 1 else ""
    print(f"  {i}. {day:6}: ${rev:.2f}{marker}")

rev_lunch = sum(o.total_bill for o in orders if o.time == "Lunch")
rev_dinner = sum(o.total_bill for o in orders if o.time == "Dinner")
cnt_lunch = sum(1 for o in orders if o.time == "Lunch")
cnt_dinner = sum(1 for o in orders if o.time == "Dinner")

print(f"\nОБІД vs ВЕЧЕРЯ")
print(f"  Обід:   {cnt_lunch:3} чеки | ${rev_lunch:.2f}  | avg ${rev_lunch/cnt_lunch:.2f}")
print(f"  Вечеря: {cnt_dinner:3} чеки | ${rev_dinner:.2f} | avg ${rev_dinner/cnt_dinner:.2f}")
print(f"  Переможець: {'Вечеря' if rev_dinner > rev_lunch else 'Обід'}")

avg_size = total_guests / len(orders)
large_cnt = sum(1 for o in orders if o.size >= 5)

print(f"\nСТОЛИ")
print(f"  Середній розмір: {avg_size:.1f} осіб")
print(f"  Великих (5+):    {large_cnt}")

top3 = sorted(orders, key=lambda o: o.total_bill, reverse=True)[:3]
print(f"\nТОП-3 ЧЕКИ")
for i, o in enumerate(top3, 1):
    tip_pct = (o.tip / o.total_bill) * 100
    print(f"  {i}. ${o.total_bill:.2f} | {o.day} {o.time} | {o.size} осіб | tip {tip_pct:.1f}%")

print("\n" + "=" * 60)

assert abs((rev_lunch + rev_dinner) - total_revenue) < 0.01
assert cnt_lunch + cnt_dinner == len(orders)
assert abs(sum(revenue_by_day.values()) - total_revenue) < 0.01
print("OK — обід+вечеря і сума по днях збігаються із загальним доходом")

        ЗВІТ ДЛЯ ВЛАСНИКА РЕСТОРАНУ

ЗАГАЛЬНА СТАТИСТИКА
  Всього чеків:         244
  Всього гостей:        627
  Загальний дохід:      $4827.77
  Загальні чайові:      $731.58
  Середній чек:         $19.79
  Середній tip %:       16.1%

НАЙКРАЩІ ДНІ
  1. Sat   : $1778.40  <- найкращий!
  2. Sun   : $1627.16
  3. Thur  : $1096.33
  4. Fri   : $325.88

ОБІД vs ВЕЧЕРЯ
  Обід:    68 чеки | $1167.47  | avg $17.17
  Вечеря: 176 чеки | $3660.30 | avg $20.80
  Переможець: Вечеря

СТОЛИ
  Середній розмір: 2.6 осіб
  Великих (5+):    9

ТОП-3 ЧЕКИ
  1. $50.81 | Sat Dinner | 3 осіб | tip 19.7%
  2. $48.33 | Sat Dinner | 4 осіб | tip 18.6%
  3. $48.27 | Sat Dinner | 4 осіб | tip 13.9%

OK — обід+вечеря і сума по днях збігаються із загальним доходом


Цей самий звіт легко перетворити на дашборд (`matplotlib`/`seaborn`: стовпчики по днях, обід vs вечеря, розподіл чеків) — тут це поза межами уроку, лише згадка.

## 🎯 TRANSFER: 5 самостійних задач на реальних даних

Той самий `orders`, тільки `for`/`dict`/`list`/`set`/comprehensions — без pandas. Розв'язки заховані під `# BEGIN SOLUTION` / `# END SOLUTION` — спробуй спочатку сам.

### Задача 1 — tip% по днях, відсортовано

Порахуй середній tip% для кожного дня (`{день: avg tip%}`), виведи відсортованим за спаданням.

In [28]:
def tip_pct_report(orders: list) -> dict:
    """Повертає {день: середній tip%}."""
    # YOUR CODE HERE
    # BEGIN SOLUTION
    by_day = defaultdict(list)
    for o in orders:
        by_day[o.day].append(o)
    return {
        day: sum((o.tip / o.total_bill) * 100 for o in day_orders) / len(day_orders)
        for day, day_orders in by_day.items()
    }
    # END SOLUTION


result = tip_pct_report(orders)
print("День   | avg tip%")
for day, pct in sorted(result.items(), key=lambda x: x[1], reverse=True):
    print(f"{day:6} | {pct:.2f}%")

assert round(result["Fri"], 2) == 16.99
assert round(result["Sun"], 2) == 16.69
print("OK")

День   | avg tip%
Fri    | 16.99%
Sun    | 16.69%
Thur   | 16.13%
Sat    | 15.32%
OK


### Задача 2 — чоловіки vs жінки

Порівняй середній чек і середній tip% для `sex == "Male"` і `sex == "Female"`.

In [29]:
def sex_report(orders: list, sex: str) -> tuple:
    """Повертає (кількість чеків, середній чек, середній tip%) для заданої статі."""
    # YOUR CODE HERE
    # BEGIN SOLUTION
    subset = [o for o in orders if o.sex == sex]
    avg_bill = sum(o.total_bill for o in subset) / len(subset)
    avg_tip_pct = sum((o.tip / o.total_bill) * 100 for o in subset) / len(subset)
    return len(subset), avg_bill, avg_tip_pct
    # END SOLUTION


for sex in ("Male", "Female"):
    count, avg_bill_sex, avg_tip_pct_sex = sex_report(orders, sex)
    print(f"{sex:6}: {count:3} чеків | avg ${avg_bill_sex:.2f} | avg tip {avg_tip_pct_sex:.2f}%")

male_count, male_avg_bill, _ = sex_report(orders, "Male")
female_count, female_avg_bill, _ = sex_report(orders, "Female")
assert male_count == 157
assert female_count == 87
assert round(male_avg_bill, 2) == 20.74
assert round(female_avg_bill, 2) == 18.06
print("OK")

Male  : 157 чеків | avg $20.74 | avg tip 15.77%
Female:  87 чеків | avg $18.06 | avg tip 16.65%
OK


### Задача 3 — аналіз по розміру столу

Для кожного `size` (1–6) — кількість чеків і середній чек **на людину** (`total_bill / size`).

In [30]:
def size_report(orders: list) -> dict:
    """Повертає {size: (кількість чеків, середній чек на людину)}."""
    # YOUR CODE HERE
    # BEGIN SOLUTION
    by_size = defaultdict(list)
    for o in orders:
        by_size[o.size].append(o)
    return {
        size: (len(size_orders), sum(o.total_bill / o.size for o in size_orders) / len(size_orders))
        for size, size_orders in by_size.items()
    }
    # END SOLUTION


result = size_report(orders)
print("Розмір | Чеків | Avg на людину")
for size in sorted(result):
    count, avg_per_person = result[size]
    print(f"{size:6} | {count:5} | ${avg_per_person:.2f}")

assert result[2][0] == 156
assert round(result[1][1], 2) == 7.24
print("OK")

Розмір | Чеків | Avg на людину
     1 |     4 | $7.24
     2 |   156 | $8.22
     3 |    38 | $7.76
     4 |    37 | $7.15
     5 |     5 | $6.01
     6 |     4 | $5.80
OK


### Задача 4 — топ-5 найщедріших клієнтів

5 чеків із найвищим `tip%` (саме відсотком, не сумою чайових).

In [31]:
def top_tippers(orders: list, n: int = 5) -> list:
    """Повертає n чеків із найвищим tip%, відсортованих за спаданням."""
    # YOUR CODE HERE
    # BEGIN SOLUTION
    return sorted(orders, key=lambda o: o.tip / o.total_bill, reverse=True)[:n]
    # END SOLUTION


top5 = top_tippers(orders)
print("День   | Час     | Сума   | tip%")
for o in top5:
    tip_pct = (o.tip / o.total_bill) * 100
    print(f"{o.day:6} | {o.time:7} | ${o.total_bill:6.2f} | {tip_pct:.1f}%")

assert len(top5) == 5
assert round((top5[0].tip / top5[0].total_bill) * 100, 1) == 71.0
print("OK")

День   | Час     | Сума   | tip%
Sun    | Dinner  | $  7.25 | 71.0%
Sun    | Dinner  | $  9.60 | 41.7%
Sat    | Dinner  | $  3.07 | 32.6%
Sat    | Dinner  | $ 11.61 | 29.2%
Sun    | Dinner  | $ 23.17 | 28.1%
OK


### Задача 5* — матриця день × час

`{день: {"Lunch": дохід, "Dinner": дохід}}` — вкладений `defaultdict(lambda: defaultdict(float))`.

In [32]:
def day_time_matrix(orders: list) -> dict:
    """Повертає {день: {час: дохід}}."""
    # YOUR CODE HERE
    # BEGIN SOLUTION
    matrix = defaultdict(lambda: defaultdict(float))
    for o in orders:
        matrix[o.day][o.time] += o.total_bill
    return matrix
    # END SOLUTION


matrix = day_time_matrix(orders)
print("День   | Lunch     | Dinner")
for day in matrix:
    lunch = matrix[day].get("Lunch", 0)
    dinner = matrix[day].get("Dinner", 0)
    print(f"{day:6} | ${lunch:8.2f} | ${dinner:8.2f}")

matrix_total = sum(v for day_totals in matrix.values() for v in day_totals.values())
assert abs(matrix_total - total_revenue) < 0.01
print("OK — сума матриці збігається із загальним доходом")

День   | Lunch     | Dinner
Sun    | $    0.00 | $ 1627.16
Sat    | $    0.00 | $ 1778.40
Thur   | $ 1077.55 | $   18.78
Fri    | $   89.92 | $  235.96
OK — сума матриці збігається із загальним доходом


## Далі

Урок 8 («Функції») бере звіт кафе за днями — виторг, кількість чеків, середній чек, найкращий день, групування за прийомом їжі — і розкладає його на функції `count_by_day`, `revenue_by_day`, `average_by_day`, `best_day`, `bills_by_time`, `print_report`: крок за кроком, з перевіркою на кожному кроці, аж до збігу виводу «до» і «після» рефакторингу символ у символ.